# BASELINE GRASP

### Imports

In [78]:
import random 
import time
from dataclasses import dataclass, field
from typing import List, Tuple, Set

import numpy as np

from baseline.POSSIBLE import DISTRICTS_POINTS as DP


### Parâmetros do Problema

In [79]:
NUMBER_AMBUS_TYPE_A = 1
NUMBER_AMBUS_TYPE_B = 4
RAIO = 0.0084
#10min 15min, 30min, 1h
#0.0056 0.0084, 0.0168, 0.0336
BONUS_TYPE_A = 1.20
TOTAL_AMBUS = NUMBER_AMBUS_TYPE_A + NUMBER_AMBUS_TYPE_B

### PARÂMETROS DO GRASP

In [80]:
MAX_ITERATIONS = 500
ALPHA = 0.20
LOCAL_SEARCH_ITER = 30 # Tentativas de melhora na busca local

### ESTRUTURA DOS DADOS

In [81]:
NUMBER_LOCATIONS = len(DP)
_keys = list(DP.keys())

COORDS_X = np.array([DP[k][0] for k in _keys])
COORDS_Y = np.array([DP[k][1] for k in _keys])
WEIGHTS = np.array([float(DP[k][2]) if str(DP[k][2]) != "nan" else 0.0 for k in _keys])
TOTAL_DEMAND = float(WEIGHTS.sum())

@dataclass
class Solution:
    positions_A: List[int] = field(default_factory=list)   # índices TypeA
    positions_B: List[int] = field(default_factory=list)   # índices TypeB
    fitness: float = 0.0
    coverage: float = 0.0   # % real de demanda coberta

    def all_positions(self) -> List[int]:
        return self.positions_A + self.positions_B

    def __lt__(self, other): return self.fitness < other.fitness
    def __gt__(self, other): return self.fitness > other.fitness


### Pré-computação da matriz de distâncias

In [82]:
def build_distance_matrix() -> np.array:
    diff_x = COORDS_X[:, None] - COORDS_X[None, :] 
    diff_y = COORDS_Y[:, None] - COORDS_Y[None, :]
    return np.sqrt(diff_x**2 + diff_y**2)

### AVALIAÇÃO

In [83]:
def evaluate(sol: Solution, dist_matrix: np.ndarray) -> None:
    covered_A: Set[int] = set()
    covered_B: Set[int] = set()

    for a in sol.positions_A:
        covered_A.update(np.where(dist_matrix[a] <= RAIO)[0].tolist())
    for b in sol.positions_B:
        covered_B.update(np.where(dist_matrix[b] <= RAIO)[0].tolist())
    
    fitness = float((WEIGHTS[list(covered_A)] * BONUS_TYPE_A).sum())
    only_B = covered_B - covered_A
    if only_B:
        fitness += float(WEIGHTS[list(only_B)].sum())

    all_covered = covered_A | covered_B
    sol.fitness = fitness
    sol.coverage = float(WEIGHTS[list(all_covered)].sum() / TOTAL_DEMAND * 100.0)


def margianal_gain(candidate: int, already_covered: Set[int], dist_matrix: np.ndarray, is_type_a: bool) -> float:
    new_covered = set(np.where(dist_matrix[candidate] <= RAIO)[0].tolist())
    new_points = new_covered - already_covered
    if not new_points: 
        return 0.0
    bonus = BONUS_TYPE_A if is_type_a else 1.0
    return float(WEIGHTS[list(new_points)].sum() * bonus)

### Construtor GRASP

In [84]:
def grasp_construction(dist_matrix: np.ndarray) -> Solution:
    """
    Constrói uma solução via Lista de Candidatos Restrita com alpha fixo, 
    sem qualquer informação externa. É a fase de construção pura do GRASP.       
    """

    sol = Solution()
    covered: Set[int] = set()
    used: Set[int] = set()

    # Aloca TypeA 
    for _ in range(NUMBER_AMBUS_TYPE_A):
        gains = [
            (loc, margianal_gain(loc, covered, dist_matrix, is_type_a=True))
            for loc in range(NUMBER_LOCATIONS) if loc not in used
        ]
        if not gains:
            break

        max_g = max(g for _, g in gains)
        min_g = min(g for _, g in gains)
        threshold = max_g - ALPHA * (max_g - min_g)
        rcl = [loc for loc, g in gains if g >= threshold]
        chosen = random.choice(rcl)

        sol.positions_A.append(chosen)
        used.add(chosen)
        covered.update(np.where(dist_matrix[chosen] <= RAIO)[0].tolist())

    for _ in range(NUMBER_AMBUS_TYPE_B):
        gains = [
            (loc, margianal_gain(loc, covered, dist_matrix, is_type_a=False))
            for loc in range(NUMBER_LOCATIONS) if loc not in used
        ]

        if not gains:
            break

        max_g = max(g for _, g in gains)
        min_g = min(g for _, g in gains)
        threshold = max_g - ALPHA * (max_g - min_g)
        rcl = [loc for loc, g in gains if g >= threshold]
        chosen = random.choice(rcl)

        sol.positions_B.append(chosen)
        used.add(chosen)
        covered.update(np.where(dist_matrix[chosen] <= RAIO)[0].tolist())

    evaluate(sol, dist_matrix)
    return sol

### BUSCA LOCAL (VIZINHANÇA COMPLETA)

In [85]:
def local_search(sol: Solution, dist_matrix: np.ndarray) -> Solution:
    """
    Vizinhança de uma solução: todas as soluções obtidas trocando
    uma ambulância de sua posicão atual para qualquer outra posição livre.

    Para cada ambulância (TypeA, TypeB), avalia todas as posições
    candidatas e executa a melhor troca encontrada. Repete até não haver 
    nenhuma melhora possível em toda a vizinhança.
    """

    best = Solution(
        positions_A=sol.positions_A[:],
        positions_B=sol.positions_B[:],
        fitness=sol.fitness,
        coverage=sol.coverage
    )

    improved = True
    while improved:
        improved = False

        # Vizinhança TypeA
        for idx in range(len(best.positions_A)):
            best_candidate = -1
            best_fitness = best.fitness
            occupied = set(best.positions_A) | set(best.positions_B)

            for candidate in range(NUMBER_LOCATIONS):
                if candidate in occupied:
                    continue
                new_A = best.positions_A[:]
                new_A[idx] = candidate
                trial = Solution(
                    positions_A=new_A,
                    positions_B=best.positions_B[:]
                )
                evaluate(trial, dist_matrix)
                if trial.fitness > best_fitness:
                    best_fitness = trial.fitness
                    best_candidate = candidate
                
            if best_candidate != -1:
                new_A = best.positions_A[:]
                new_A[idx] = best_candidate
                best = Solution(
                    positions_A=new_A,
                    positions_B=best.positions_B[:]
                )
                evaluate(best, dist_matrix)
                improved = True
        
        # Vizinhança TypeB
        for idx in range(len(best.positions_B)):
            best_candidate = -1
            best_fitness = best.fitness
            occupied = set(best.positions_A) | set(best.positions_B)

            for candidate in range(NUMBER_LOCATIONS):
                if candidate in occupied:
                    continue
                new_B = best.positions_B[:]
                new_B[idx] = candidate
                trial = Solution(
                    positions_A=best.positions_A[:],
                    positions_B=new_B
                )
                evaluate(trial, dist_matrix)
                if trial.fitness > best_fitness:
                    best_fitness = trial.fitness
                    best_candidate = candidate

            if best_candidate != -1:
                new_B = best.positions_B[:]
                new_B[idx] = best_candidate
                best = Solution(
                    positions_A=best.positions_A[:],
                    positions_B=new_B
                )
                evaluate(best, dist_matrix)
                improved = True
        
    return best

### FUNÇÃO PRINCIPAL

In [86]:
def run(seed: int = None, verbose: bool=False) -> Tuple[Solution, List[float]]:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    dist_matrix = build_distance_matrix()
    print(f"Shape esperado: (3016, 3016)")
    print(f"Shape real:     {dist_matrix.shape}")
    best_sol: Solution = None
    history: List[float] = []

    for it in range(MAX_ITERATIONS):
        sol = grasp_construction(dist_matrix)
        sol = local_search(sol, dist_matrix)

        if best_sol is None or sol.fitness > best_sol.fitness:
            best_sol = sol

        history.append(best_sol.coverage)

        if verbose and it % 50 == 0:
            print(f" GRASP Iter {it:4d} | Cobertura: {best_sol.coverage:.2f}%")

    return best_sol, history

if __name__ == "__main__":
    print("===== GRASP =====")
    t0 = time.time()
    best_sol, hist = run(seed=42, verbose=True)
    elapsed = time.time() - t0
    print(f"\nMelhor cobertura: {best_sol.coverage:.2f}%")
    print(f"Tempo: {elapsed:.1f}s")
    print(f"Posições TypeA: {best_sol.positions_A}")
    print(f"Posições TypeB: {best_sol.positions_B}")


===== GRASP =====
Shape esperado: (3016, 3016)
Shape real:     (3016, 3016)
 GRASP Iter    0 | Cobertura: 40.47%
 GRASP Iter   50 | Cobertura: 40.80%
 GRASP Iter  100 | Cobertura: 40.80%
 GRASP Iter  150 | Cobertura: 40.80%


KeyboardInterrupt: 